# LA Airbnb cleaning checks

This notebook checks the cleaning output against the original listings data.

In [1]:
import pandas as pd

raw = pd.read_csv("LA_listings.csv")
cleaned = pd.read_csv("LA_listings_cleaned_yaxin.csv")

print("Original:", raw.shape)
print("Cleaned:", cleaned.shape)

Original: (43932, 19)
Cleaned: (43932, 21)


In [2]:
ids = [2708, 2732, 6033, 9140, 584724]

before = raw.loc[raw["id"].isin(ids), ["id", "price"]].copy()
before["expected_has_price"] = before["price"].notna()
before["expected_high_price_review"] = before["price"] > 1000

after = cleaned.loc[
    cleaned["id"].isin(ids),
    ["id", "has_price", "high_price_review"]
]

check = before.merge(after, on="id")
check["has_price_matches"] = check["expected_has_price"] == check["has_price"]
check["high_price_matches"] = (
    check["expected_high_price_review"] == check["high_price_review"]
)
check

,id,price,expected_has_price,expected_high_price_review,has_price,high_price_review,has_price_matches,high_price_matches
0,2708,67.0,True,False,True,False,True,True
1,2732,214.0,True,False,True,False,True,True
2,6033,100.0,True,False,True,False,True,True
3,9140,NaN,False,False,False,False,True,True
4,584724,1043.0,True,True,True,True,True,True


## Cleaning decisions and checks

| Decision | Reason | Check |
|---|---|---|
| Keep rows with missing price; add `has_price` | A missing price should not be invented, and the listing may still be useful for other analyses. | 5,570 prices are missing; ID 9140 has `has_price = False`. |
| Keep prices above $1,000; add `high_price_review` | An unusually high price needs review, but it is not enough evidence to delete a listing. | 2,493 listings are flagged; ID 584724 ($1,043) is `True`. |
| Keep original listing IDs | IDs are needed to match original and processed records. | 0 duplicate IDs; all five checked IDs appear in the output. |
| Keep `availability_365` values | No values failed the 0–365 range check. | 0 invalid values. |

The five record checks above compare expected flags from the original price with the actual flags in the processed file. Both comparisons match for all five records.

In [3]:
print("Exact duplicate rows:", raw.duplicated().sum())
print("Repeated listing IDs:", raw["id"].duplicated().sum())

Exact duplicate rows: 0
Repeated listing IDs: 0


In [4]:
print(raw.dtypes)

id                                  int64
name                               object
host_id                           float64
host_profile_id                   float64
host_name                          object
neighbourhood_group                object
neighbourhood                      object
latitude                          float64
longitude                         float64
room_type                          object
price                             float64
minimum_nights                    float64
number_of_reviews                   int64
last_review                        object
reviews_per_month                 float64
calculated_host_listings_count    float64
availability_365                    int64
number_of_reviews_ltm               int64
license                            object
dtype: object


In [5]:
import pandas as pd

missing_check = pd.DataFrame({
    "before": raw.isna().sum(),
    "after": cleaned[raw.columns].isna().sum()
})
display(missing_check[missing_check["before"] > 0])

,before,after
host_id,181,181
host_profile_id,395,395
host_name,384,384
price,5570,5570
minimum_nights,11,11
last_review,11311,11311
reviews_per_month,11311,11311
calculated_host_listings_count,181,181
license,30740,30740


In [6]:
dates = pd.to_datetime(raw["last_review"], errors="coerce")
invalid_dates = raw["last_review"].notna() & dates.isna()
print("Nonempty invalid last_review dates:", invalid_dates.sum())

Nonempty invalid last_review dates: 0


In [7]:
names = raw["neighbourhood"].dropna()
extra_spaces = names.ne(names.str.strip())
print("Neighbourhood names with extra spaces:", extra_spaces.sum())

Neighbourhood names with extra spaces: 0


In [8]:
print("All listings:", len(cleaned))
print("Listings with price:", cleaned["price"].notna().sum())
print("Listings missing price:", cleaned["price"].isna().sum())
print("Median price among listings with price:", cleaned["price"].median())

All listings: 43932
Listings with price: 38362
Listings missing price: 5570
Median price among listings with price: 224.0
